# ***IntelliHack***

## **Hackathon Problem Statements On AI in Cybersecurity**

----

### *Problem Statement - 4*

**Problem Statement:** AI for Real-Time Fraud & Social Engineering Detection

***Description of the Problem:*** Industries, particularly financial services, are losing billions of dollars to social engineering scams, such as fake investment schemes and fraudulent calls. The problem is to create an AI system that can analyze live text or audio conversations to detect and warn against suspicious patterns and phrases.

* **Expectation form Prototype -**
   * An AI-driven assistant that analyzes live text/audio conversations.

   * A model that can detect urgency phrases or suspicious transaction  patterns.

   * A real-time warning system to alert users of potential fraud.




### **Prototype Name :-**

#### **AlertX – Emphasizes Financial Security**

*Built-up by AI Devilops*

Team Lead :- Deepak Kaura

Other member :- Vivek Nayi

* ***Note - It's all about BERT Classiication (NLP) + LLM integration also includes Gradio (deployment)***

In [ ]:
#import pandas as pd


#data = pd.read_csv("fraud_call.file",sep='\t',names=['label','content'], on_bad_lines='skip')

In [ ]:
#data.to_csv('fraud call fin.csv', index=False)

## **Loading Important Libraries**

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import BertTokenizer, BertModel
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
#from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score, confusion_matrix
import plotly.figure_factory as ff



data_df = pd.read_csv('fraud call fin.csv')

### **Checking first 5 rows**

In [ ]:
data_df.head()

,label,content
0,1,"hello, i m bank manager of SBI, ur debit card ..."
1,1,Todays Vodafone numbers ending with 4882 are s...
2,0,Please don't say like that. Hi hi hi
3,0,Thank you!
4,0,Oh that was a forwarded message. I thought you...


### **Checking each column/feature datatype**

In [ ]:
data_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5924 entries, 0 to 5923
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    5924 non-null   int64 
 1   content  5924 non-null   object
dtypes: int64(1), object(1)
memory usage: 92.7+ KB


### **Checking target column value counts**

In [ ]:
data_df.label.value_counts()

,count
label,
0,5286
1,638


## **Time To build BERT model**

In [ ]:

# ===============================
# 1. Load data
# ===============================
y = data_df['label'].values

In [ ]:
# ===============================
# 2. Load BERT model
# ===============================
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert = BertModel.from_pretrained("bert-base-uncased")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bert.to(device)
bert.eval()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [ ]:
# ===============================
# 3. Extract BERT embeddings (CLS)
# ===============================
def get_embeddings(texts, batch_size=16, max_len=128):
    all_embeds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=max_len, return_tensors="pt")
        input_ids = enc["input_ids"].to(device)
        attn_mask = enc["attention_mask"].to(device)
        with torch.no_grad():
            outputs = bert(input_ids, attention_mask=attn_mask)
            cls_embeds = outputs.last_hidden_state[:,0,:]  # CLS token
            all_embeds.append(cls_embeds.cpu().numpy())
    return np.vstack(all_embeds)

X = get_embeddings(data_df["content"].tolist())


In [ ]:
# ===============================
# 4. Define classifier head
# ===============================
num_labels = len(set(y))
classifier = nn.Sequential(
    nn.Linear(X.shape[1], 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, num_labels)
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(classifier.parameters(), lr=2e-5)

In [ ]:

# =====================================================
# 5. Get embeddings, labels, and raw text
# =====================================================
                   # numeric labels
texts = data_df["content"].values                # original sentences

# =====================================================
# 6. Split indices first (so everything aligns)
# =====================================================
indices = np.arange(len(X))
train_idx, test_idx = train_test_split(
    indices, test_size=0.2, stratify=y, random_state=42
)

# Slice embeddings, labels, and texts with the same indices
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
text_train, text_test = texts[train_idx], texts[test_idx]

# =====================================================
# 7. Apply SMOTE on training embeddings only
# =====================================================
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# =====================================================
# 8. Create DataLoaders
# =====================================================
train_dataset = TensorDataset(torch.tensor(X_train_res, dtype=torch.float32),
                              torch.tensor(y_train_res, dtype=torch.long))

test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32),
                             torch.tensor(y_test, dtype=torch.long))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# =====================================================
# 9. Training loop
# =====================================================
epochs = 5
for epoch in range(epochs):
    classifier.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = classifier(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}")

# =====================================================
# 10. Evaluation loop
# =====================================================
classifier.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = classifier(xb)
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(yb.cpu().numpy())

# =====================================================
# 11. Accuracy
# =====================================================
print('----' * 16)
print(f"✅ BERT Classifier Accuracy: {accuracy_score(all_labels, all_preds):.2f}")
print('----' * 16)

# =====================================================
# 12. Confusion Matrix
# =====================================================
cm = confusion_matrix(all_labels, all_preds)

# If you have label names, use them here
labels = ['Normal', 'Farud']   # <-- replace with your class names
z_text = [[str(y) for y in x] for x in cm]

fig = ff.create_annotated_heatmap(
    z=cm,
    x=labels,  # Predicted
    y=labels,  # True
    annotation_text=z_text,
    colorscale='Blues',
    showscale=True
)

fig.update_layout(
    title_text='Confusion Matrix - BERT Classifier',
    width=550,
    height=500
)

fig['data'][0]['showscale'] = True
fig.show()



Epoch 1/5 - Loss: 0.0793
Epoch 2/5 - Loss: 0.0724
Epoch 3/5 - Loss: 0.0680
Epoch 4/5 - Loss: 0.0643
Epoch 5/5 - Loss: 0.0620
----------------------------------------------------------------
✅ BERT Classifier Accuracy: 0.96
----------------------------------------------------------------


In [ ]:
# =====================================================
# 13. Build results DataFrame with original content
# =====================================================
results_llm = pd.DataFrame({
    "content": text_test,          # original text (aligned by indices)

    "predicted_label": all_preds   # predictions
})

results_llm.head()

,content,predicted_label
0,you have a secret admirer. REVEAL who thinks y...,1
1,He is impossible to argue with and he always t...,0
2,Shuhui has bought ron's present it's a swatch ...,0
3,"Aight I've been set free, think you could text...",0
4,"Cool, we shall go and see, have to go to tip a...",0


In [ ]:
results_llm.to_csv('results_llm_fraud_calls.csv', index=False)

## **Time for utilizing Test data on LLM**

In [ ]:
import pandas as pd

results_llm_df = pd.read_csv('results_llm_fraud_calls.csv')

In [ ]:
# Apply categorization directly on results_llm
def categorize_sms(text, label):
    if label == 0:
        return "Normal SMS"

    text = str(text).lower()
    if any(w in text for w in ["win", "prize", "reward", "lottery", "lucky", "cash"]):
        return "Lottery Scam"
    elif any(w in text for w in ["loan", "bank", "credit", "account", "card", "insurance"]):
        return "Financial Scam"
    elif any(w in text for w in ["love", "admirer", "dating", "chat", "sexy"]):
        return "Romance Scam"
    elif any(w in text for w in ["free", "ringtone", "offer", "call", "sms"]):
        return "Promotion/Spam"
    else:
        return "Other Fraud"

# Create a new column with category labels
results_llm_df["category"] = results_llm_df.apply(
    lambda row: categorize_sms(row["content"], row["predicted_label"]), axis=1
)

# Preview
print(results_llm_df.head(10))


                                             content  predicted_label  \
0  you have a secret admirer. REVEAL who thinks y...                1   
1  He is impossible to argue with and he always t...                0   
2  Shuhui has bought ron's present it's a swatch ...                0   
3  Aight I've been set free, think you could text...                0   
4  Cool, we shall go and see, have to go to tip a...                0   
5  It’s been really long we guys haven’t gone any...                0   
6    I finished my lunch already. U wake up already?                0   
7  No de. But call me after some time. Ill tell y...                0   
8  Awesome question with a cute answer: Someone a...                0   
9           Is this the house you wanted to show us?                0   

       category  
0  Romance Scam  
1    Normal SMS  
2    Normal SMS  
3    Normal SMS  
4    Normal SMS  
5    Normal SMS  
6    Normal SMS  
7    Normal SMS  
8    Normal SMS  
9    Normal SMS 

In [ ]:
!pip install langchain

In [ ]:
!pip install txtai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 MB 20.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 9.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Using cached ninja-1.13.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 999.8/999.8 kB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/224.9 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!pip install langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [ ]:

from txtai import LLM

txtai = LLM("MaziyarPanahi/gemma-2-2b-it-GGUF/gemma-2-2b-it.Q8_0.gguf")



[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package cmudict to /root/nltk_data...
[nltk_data]   Unzipping corpora/cmudict.zip.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


gemma-2-2b-it.Q8_0.gguf:   0%|          | 0.00/2.78G [00:00<?, ?B/s]

llama_kv_cache_unified_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


## ***Deployment : Gradio***

*Final stage: Testing our LLM-powered alert system with real data via the UI*

In [ ]:
import gradio as gr
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.schema import StrOutputParser
from txtai.pipeline import LLM




llm = TxtaiLangChainLLM()

# ===============================
# 2. Rule-based fraud categorizer
# ===============================
def categorize_sms(text):
    text = str(text).lower()

    if any(w in text for w in ["win", "prize", "reward", "lottery", "lucky", "cash"]):
        return "Lottery Scam"
    elif any(w in text for w in ["loan", "bank", "credit", "account", "card", "insurance"]):
        return "Financial Scam"
    elif any(w in text for w in ["love", "admirer", "dating", "chat", "sexy"]):
        return "Romance Scam"
    elif any(w in text for w in ["free", "ringtone", "offer", "call", "sms"]):
        return "Promotion/Spam"
    elif any(w in text for w in ["customer care", "support", "helpline", "service center", "toll free", "call us"]):
        return "Customer Care Scam"
    else:
        return "Normal SMS"

# ===============================
# 3. Rule-based Alert Messages
# ===============================
def fraud_alert_message(content, category):
    if category == "Normal SMS":
        return "✅ This SMS appears safe and normal."
    elif category == "Lottery Scam":
        return f"⚠️ Fraud Alert: Lottery Scam.\nSMS: {content}\nRecommended Action: Ignore and delete. Never share personal or financial details."
    elif category == "Financial Scam":
        return f"⚠️ Fraud Alert: Financial Scam.\nSMS: {content}\nRecommended Action: Do not click any links or share banking details."
    elif category == "Romance Scam":
        return f"⚠️ Fraud Alert: Romance Scam.\nSMS: {content}\nRecommended Action: Do not respond. Scammers may attempt emotional manipulation."
    elif category == "Promotion/Spam":
        return f"⚠️ Fraud Alert: Promotion/Spam.\nSMS: {content}\nRecommended Action: Block the sender. Do not call or subscribe."
    elif category == "Customer Care Scam":
        return f"⚠️ Fraud Alert: Customer Care Scam.\nSMS: {content}\nRecommended Action: Do not call or trust fake helpline numbers. Always verify from official sources."
    else:
        return f"⚠️ Unknown Category.\nSMS: {content}\nPlease investigate further."

# ===============================
# 4. LLM Explanation
# ===============================
fraud_explain_prompt = PromptTemplate.from_template("""
You are an SMS fraud detection assistant.

SMS Content: {content}
Detected Category: {category}

Explain briefly (in 1-2 sentences) **why this SMS belongs to this fraud category**.
Keep the explanation under 30 words.
""")

fraud_explain_chain = LLMChain(
    llm=llm,
    prompt=fraud_explain_prompt,
    output_parser=StrOutputParser()
)

# ===============================
# 5. Backend Logic
# ===============================
def analyze_sms(user_sms):
    category = categorize_sms(user_sms)
    alert_message = fraud_alert_message(user_sms, category)

    if category != "Normal SMS":
        explanation = fraud_explain_chain.run({
            "content": user_sms,
            "category": category
        })
    else:
        explanation = "This SMS is normal and safe."

    return category, alert_message, explanation

# ===============================
# 6. Gradio UI
# ===============================
with gr.Blocks() as demo:
    gr.Markdown("## 📩 AlertX – Emphasizes Financial Security")

    with gr.Row():
        with gr.Column(scale=1):
            sms_input = gr.Textbox(
                label="Enter SMS Content",
                placeholder="Paste SMS text here...",
                lines=4
            )
            analyze_btn = gr.Button("🔍 Analyze SMS")

        with gr.Column(scale=2):
            category_out = gr.Textbox(label="Fraud Category")
            alert_out = gr.Textbox(label="Alert Message", lines=5)
            explain_out = gr.Textbox(label="Explanation", lines=3)

    analyze_btn.click(
        analyze_sms,
        inputs=[sms_input],
        outputs=[category_out, alert_out, explain_out]
    )

demo.launch()


llama_kv_cache_unified_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
/tmp/ipython-input-3538800967.py:74: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  fraud_explain_chain = LLMChain(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3f1797920b77ea4da8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
